# Setup

In [23]:
import os, json, subprocess, time, threading, gc, sys, re, uuid, math

from typing import Annotated, List, TypedDict, Union, Dict, Literal, Any, Optional, Union

# Langchain
from langchain_ollama import ChatOllama
from langchain_qdrant import QdrantVectorStore
from langchain.tools import tool

from langchain_core.documents import Document
from langchain_core.tools import StructuredTool
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage, AIMessageChunk, trim_messages
from langchain_core.documents import Document
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

# Qdrant
from qdrant_client import QdrantClient
from qdrant_client.http import models

# Langgraph
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# Langsmith
import langsmith as ls
from langsmith import Client

# PER TOOL STRUTTURATI
from pydantic import create_model, Field, BaseModel
from qdrant_client import models

# Interfaccia
import gradio as gr

In [24]:
# Imposta le tue chiavi API
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = "inserire_chiave"


# Opzionale: specifica il nome del progetto
os.environ["LANGCHAIN_PROJECT"] = "mio-progetto-qwen"


In [25]:
project_name = "mio-progetto-qwen"

with ls.trace(
    "TEST manuale da Colab",
    "chain",
    project_name=project_name,
    inputs={"test": "ciao"}
) as rt:
    rt.end(outputs={"risultato": "LangSmith funziona"})

Client().flush()

print("Trace manuale inviata al progetto:", project_name)

Trace manuale inviata al progetto: mio-progetto-qwen


In [26]:
current_dir = os.getcwd()

DB_PATH = os.path.join(current_dir, "qdrant_db")

qdrant_lock = threading.Lock()

In [27]:
def sblocca_qdrant_breve():
    # 1. Chiude esplicitamente i client di primo livello (se esistono)
    if 'client' in globals(): globals()['client'].close()
    if 'vectorstore' in globals(): globals()['vectorstore'].client.close()
    
    # 2. Identifica le variabili da eliminare + la cronologia nascosta di Colab (_, _1, _2...)
    da_rimuovere = ['client', 'vectorstore', 'retrievers_map'] + [k for k in globals() if k.startswith('_')]
    
    # 3. Rimuove tutto in un colpo solo
    for k in da_rimuovere:
        globals().pop(k, None)
        
    # 4. Forza la pulizia: questo distruggerà i retriever e rilascerà i lock rimanenti
    gc.collect()
    print("✨ Pulizia completata! Lock rilasciati.")

sblocca_qdrant_breve()

✨ Pulizia completata! Lock rilasciati.


In [28]:
client = QdrantClient(path = DB_PATH)

# Descrizioni Collezioni

In [29]:
description_contatti_docenti = """
Strumento principale per accedere a tutte le informazioni riguardanti numeri telefonici, email, orari di ricevimento, oltre che per rispondere a domande sulle competenze, la carriera e l'attività didattica dei professori.
Utilizza questo tool se l'utente chiede:
- L'indirizzo email, l'indirizzo fisico o il numero di telefono di un docente e di un ufficio.
- Dove si trova fisicamente un ufficioo il dipartimento e come comunicare con la Segreteria didattica o il personale tecnico-amministrativo.
- Il curriculum vitae o il percorso di studi di un docente.
- Quali corsi insegna un professore (es. 'Chi insegna NLP?').
- Competenze tecniche specifiche (es. 'Chi si occupa di Computer Vision o Deep Learning?').
"""

description_international = """
Strumento principale per trovare tutte le informazioni relative all'internazionalizzazione, alla mobilità estera e agli accordi internazionali del Dipartimento (DIEM). Utilizza questo tool se l'utente chiede:
- Accordi Erasmus Plus (inclusa la mobilità per studio, traineeship/tirocinio e docenza).
- Elenco e dettagli degli Accordi di Cooperazione Internazionale stipulati con istituzioni partner.
- Opportunità e accordi per il conseguimento del Doppio Titolo (Double Degree).
- Informazioni sui Dottorati con Tesi in Co-Tutela con università estere.
- Dettagli su International Credit Mobility e programmi per Visiting Professors.
- Informazioni generali e dettagli sui Bandi Erasmus+.
- Contatti e riferimenti specifici per la mobilità internazionale e l'internazionalizzazione della didattica.
"""

description_offerta_formativa = """
Strumento principale per fornire informazioni sull'Offerta Formativa del Dipartimento di Ingegneria dell'Informazione ed Elettrica e Matematica applicata (DIEM).

Utilizza questo tool se l'utente chiede informazioni su corsi di studio, ammissione, piani di studio, curricula, regolamenti, sbocchi professionali, tesi relativi all'offerta formativa del Dipartimento.
Utilizza questo tool per informazioni riguardanti l'ubicazione delle aule.
Utilizza questo tool per informazioni su accesso alle TRIENNALI in base al TOLC e informazioni sui requisiti di accesso alle MAGISTRALI.

Il tool copre:

1. LAUREE TRIENNALI:
   Informazioni sui corsi di Laurea Triennale del Dipartimento.
   Include:
   - Ingegneria Informatica.
   - Ingegneria dell'Informazione per la Medicina Digitale.

2. LAUREE MAGISTRALI:
   Informazioni sui corsi di Laurea Magistrale del Dipartimento.
   Include:
   - Ingegneria Informatica.
   - Electrical Engineering for Digital Energy.
   - Information Engineering for Digital Medicine.

3. DOTTORATI:
   Informazioni sui programmi di Dottorato di Ricerca afferenti al Dipartimento.
   Include:
   - Dottorato in Ingegneria dell'Informazione.
   - Dottorato Nazionale in Photovoltaics.

"""

description_bandi = """
Strumento principale per trovare informazioni su concorsi, selezioni pubbliche, incarichi e opportunità di finanziamento presso il Dipartimento (DIEM). Utilizza questo tool se l'utente chiede:

1. INCARICHI DI INSEGNAMENTO: Bandi per il conferimento di contratti di insegnamento a titolo retribuito o gratuito per i vari anni accademici (es. A.A. 2025/2026).
2. RICERCA E ASSEGNI: Informazioni su selezioni per Assegni di Ricerca, Borse di Ricerca e altre collaborazioni scientifiche con il Dipartimento.
3. OPPORTUNITÀ PER STUDENTI: Bandi per borse di studio, premi di laurea, borse per tutorato (es. SADB) e altre forme di sostegno al diritto allo studio.
4. PERSONALE E COLLABORAZIONI: Selezioni per Personale Tecnico Amministrativo (PTA) e avvisi per incarichi di collaborazione esterna.
5. ESITI E GRADUATORIE: Risultati delle selezioni, verbali delle commissioni esaminatrici, graduatorie di merito e decreti di approvazione atti.
6. PROCEDURE AMMINISTRATIVE: Informazioni sulle modalità di partecipazione, scadenze, requisiti di ammissione e rinvii a portali ufficiali come l'Albo Ufficiale d'Ateneo.

Il tool restituisce dettagli estratti dagli avvisi di indizione, dai regolamenti dei concorsi e dalla documentazione amministrativa correlata.
"""

description_diem_news_mission_focus = """
Utile per ottenere informazioni aggiornate sulla vita del Dipartimento (DIEM), le news, le sue iniziative di eccellenza didattica e il suo impatto sul territorio e l'industria.
Utilizza questo tool unico se l'utente chiede:

1. NEWS ED EVENTI: Aggiornamenti su seminari scientifici (es. cybersecurity, real-time systems), convegni, incontri di orientamento per matricole, cerimonie di premiazione e avvisi istituzionali. Include calendari di eventi passati e futuri e graduatorie di ammissione a percorsi formativi.
2. FOCUS DIDATTICA E ACADEMY: Dettagli sui servizi di Tutorato per il supporto allo studio, informazioni sulla Academy DIEM e sui Percorsi di Eccellenza per lauree triennali e magistrali. Include procedure per bootcamp specialistici (es. Swift App Development Bootcamp) e certificazioni internazionali come EUR-ACE.
3. TERZA MISSIONE E TRASFERIMENTO TECNOLOGICO: Informazioni sulla valorizzazione della ricerca attraverso Spin-off (es. A.I. TECH, IPERA, AI4Health, AI-READY) e tutela della proprietà intellettuale con brevetti nazionali e internazionali.
4. IMPATTO SOCIALE E PUBBLICO IMPEGNO: Iniziative di public engagement, formazione continua (es. Polo di Salerno dell'Accademia dei Lincei) e progetti di valorizzazione del territorio e del patrimonio storico (es. mostre sul Centro storico di Salerno).
5. ALTERNANZA SCUOLA LAVORO: Dettagli sui progetti formativi FSL (ex PCTO) offerti dal Dipartimento agli studenti dell'ultimo anno delle scuole superiori per promuovere una scelta universitaria consapevole. Include informazioni e regolamenti su competizioni specifiche in continua espansione come la RobotCup@School e la DigitalMedicineCup@School .
6. RICERCA: Dettagli sulle molteplici aree tematiche indagate dai gruppi del Dipartimento, tra cui l'Ingegneria Informatica e l'Elettrotecnica. Include approfondimenti sui principali settori di innovazione e sugli obiettivi strategici esplorati. Fornisce informazioni sui fondi ottenuti tramite progetti finanziati, su riconoscimenti scientifici e accademici e rimanda al catalogo istituzionale IRIS.

Non usare il tool se vengono richieste informazioni sul TOLC
Il tool restituisce notizie recenti, brochure informative sui percorsi di eccellenza, elenchi di brevetti e dettagli su collaborazioni con il mondo industriale e sociale.
"""

description_diem_istituzionale_strutture = """
Utile per ottenere informazioni ufficiali sulla struttura, l'organizzazione, le commissioni e le infrastrutture del Dipartimento (DIEM).
NON utilizzare questo tool se l'utente chiede contatti e ubicazione fisica di edifici o organi.
Utilizza questo tool se l'utente chiede:

1. PRESENTAZIONE E GOVERNANCE: Visione d'insieme del Dipartimento, i suoi pilastri (ricerca, didattica, trasferimento tecnologico), il Direttore del Dipartimento e i riconoscimenti di eccellenza.
2. CONSIGLIO DIDATTICO: Composizione e compiti del Consiglio Didattico di Ingegneria Informatica, inclusi i nominativi del Presidente, dei membri e dei rappresentanti degli studenti.
3. COMMISSIONI E DELEGATI: Informazioni sugli organi collegiali e sulle figure di riferimento per compiti specifici, come i Gruppi di Assicurazione Qualità (GAQ), la Commissione Didattica, e i delegati.
4. LABORATORI E STRUTTURE: Elenco e finalità dei laboratori di ricerca (es. MIVIA, Robotica, Cybersecurity, Bioingegneria, Energie Rinnovabili) e dei laboratori didattici. Include dettagli sulle infrastrutture.
5. SEDI E LOGISTICA: Riferimenti alle attività presso la sede di Avellino e responsabilità logistiche associate.

Il tool restituisce dati ufficiali tratti dai verbali del consiglio, dagli organigrammi di dipartimento e dalle schede tecniche delle strutture di ricerca.
"""

Associazione delle descrizioni alle collezioni

In [30]:
collections_config = [

    {
        "collection_name": "docenti", #contatti_docenti
        "description": description_contatti_docenti
    },
    {
        "collection_name": "international",
        "description": description_international
    },

    {
        "collection_name": "offerta_formativa",
        "description": description_offerta_formativa
    },
    {
        "collection_name": "bandi",
        "description": description_bandi
    },
    {
        "collection_name": "iniziative_ricerca_alternanza",
        "description": description_diem_news_mission_focus
    },
    {
        "collection_name": "dipartimento",
        "description": description_diem_istituzionale_strutture
    }

]



# Gestione Manifest per futura inezione nel prompt del Planner

In [31]:
from pathlib import Path
from typing import Dict, List

# 1. Ricava la cartella radice del progetto (salendo di un livello rispetto al notebook attuale)
BASE_DIR = Path.cwd().parent

# 2. Definisce il percorso assoluto dinamico verso la cartella dei manifest
MANIFEST_DIR = BASE_DIR / "metadata_manifest"

# 3. Costruisce il dizionario configurando i percorsi in modo dinamico
MANIFEST_CONFIG: Dict[str, List[str]] = {
    "international": [
        str(MANIFEST_DIR / "manifest_international.json")
    ],

    "docenti": [
        str(MANIFEST_DIR / "manifest_docenti.json"),
        str(MANIFEST_DIR / "manifest_contatti.json")
    ],

    "offerta_formativa": [
        str(MANIFEST_DIR / "manifest_offerta_formativa.json")
    ],

    "bandi": [
        str(MANIFEST_DIR / "manifest_bandi.json")
    ],

    "iniziative_ricerca_alternanza": [
        str(MANIFEST_DIR / "manifest_TerzaMissione.json"),
        str(MANIFEST_DIR / "manifest_news.json"),
        str(MANIFEST_DIR / "manifest_eventi.json"),
        str(MANIFEST_DIR / "manifest_ricerca.json"),
        str(MANIFEST_DIR / "manifest_alternanza.json"),
        str(MANIFEST_DIR / "manifest_focus_didattica.json")
    ],
              
    "dipartimento": [
        str(MANIFEST_DIR / "manifest_dipartimento.json"),
        str(MANIFEST_DIR / "manifest_consiglio_didattico.json")
    ]
}

def load_all_manifests(config: Dict[str, List[str]]) -> Dict[str, List[dict]]:
    """
    Carica in memoria tutti i file JSON dei manifest associati ai tool.
    Restituisce un dizionario dove la chiave è il nome della collezione e il valore è una LISTA di dizionari.
    Gestisce i file mancanti o corrotti ignorandoli senza far fallire gli altri.
    """
    loaded_map: Dict[str, List[dict]] = {}

    for tool_name, file_paths in config.items():
        loaded_map[tool_name] = [] # Inizializza sempre la lista per il tool, anche se resterà vuota

        for file_path in file_paths:
            if os.path.exists(file_path):
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        loaded_manifest = json.load(f)
                        loaded_map[tool_name].append(loaded_manifest)
                    print(f"✅ Manifest '{file_path}' per il tool '{tool_name}' caricato correttamente.")
                except json.JSONDecodeError as e:
                    print(f"❌ Errore di sintassi JSON nel file '{file_path}': {e}")
            else:
                print(f"⚠️ Avviso: Il file manifest non è stato trovato in: {file_path}")

    return loaded_map


manifests_map: Dict[str, List[dict]] = load_all_manifests(MANIFEST_CONFIG)

✅ Manifest '/Users/mariasilvanachiarelotto/Documents/Esami/NLP&LLM/Consegna NLP&LLM/chatbot_diem/metadata_manifest/manifest_international.json' per il tool 'international' caricato correttamente.
✅ Manifest '/Users/mariasilvanachiarelotto/Documents/Esami/NLP&LLM/Consegna NLP&LLM/chatbot_diem/metadata_manifest/manifest_docenti.json' per il tool 'docenti' caricato correttamente.
✅ Manifest '/Users/mariasilvanachiarelotto/Documents/Esami/NLP&LLM/Consegna NLP&LLM/chatbot_diem/metadata_manifest/manifest_contatti.json' per il tool 'docenti' caricato correttamente.
✅ Manifest '/Users/mariasilvanachiarelotto/Documents/Esami/NLP&LLM/Consegna NLP&LLM/chatbot_diem/metadata_manifest/manifest_offerta_formativa.json' per il tool 'offerta_formativa' caricato correttamente.
✅ Manifest '/Users/mariasilvanachiarelotto/Documents/Esami/NLP&LLM/Consegna NLP&LLM/chatbot_diem/metadata_manifest/manifest_bandi.json' per il tool 'bandi' caricato correttamente.
✅ Manifest '/Users/mariasilvanachiarelotto/Document

# Parametri per ricerca nel Database e Inizializzazione modello di embedding

In [32]:
SEARCH_VECTOR_NAMES = ("vector_question_1", "vector_question_2", "")
QDRANT_LIMIT_PER_VECTOR = 15
TOP_K_FINAL = 6

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'mps'},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 3}
)

# Per isolare nomi collezioni
retrievers_map = {
    c["collection_name"]: None
    for c in collections_config
}

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [33]:
METADATA_CLAUSE_MAPPING: Dict[str, str] = {

    # Filtri Must
    "year": "must",
    "state": "must",
    "anno": "must",
    "stato": "must",
    "commission": "must",
    "membri": "must",
    "nominativo": "must",
    "prof_name": "must",
    "prof_surname": "must",
    "courses": "must",
    "titolo": "must",
    "livello": "must",
    "corso": "must",
    "curriculum": "must",
    "paese": "must",
    "universita_partner": "must",
    "posti_disponibili": "must",
    "course_type": "must",
    "course_name": "must",
    "academic_year": "must",
    "project_period": "must",
    "responsabile": "must",

    # Filtri Should (possibili must)
    "title": "should",
    "keywords": "should",
    "questions": "should",
    "dipartimento": "should",
    "ruolo": "should",
    "ufficio": "should",
    "struttura": "should",
    "titolo": "should",
    "data": "should",
    "programma": "should",
    "destinatario": "should",
    "research_area": "should",
    "nome_progetto": "should",
    "tipologia": "should"
}

# Pipeline LangGraph (Nodi e Workflow)

## Output strutturati, definizione modelli LLM, definizione AgentState, definizioni di funzioni

In [34]:
# Definizione schemi Pydantic per output strutturato

class QueryClassification(BaseModel):
    intent_type: Literal["diem_query", "use_calculator", "chit_chat", "out_of_domain", "inappropriate", "needs_clarification"] = Field(
        description="Classifica l'intento dell'utente in base alle definizioni fornite."
    )

class QueryReformulation(BaseModel):
    queries: List[str] = Field(
        description="Lista di una o più query ottimizzate, atomiche e prive di pronomi/ambiguità."
    )

class ToolAssignment(BaseModel):
    query: str = Field(description="La query ottimizzata ricevuta in input.")
    tool_name: str = Field(description="Il nome esatto della collezione selezionata per questa specifica query.")

class ToolSelection(BaseModel):
    assignments: List[ToolAssignment] = Field(
        description="Mappatura di ogni query ottimizzata con la rispettiva collezione ideale."
    )

class SearchTask(BaseModel):
    tool_name: str = Field(description="Il nome della collezione dove cercare.")
    query: Optional[str] = Field(default=None, description="Query riformulata/ottimizzata usata per la ricerca.")
    filters: List[Dict[str, Any]] = Field(default_factory=list, description="Filtri estratti: key, value, operator.")

class ExecutionPlan(BaseModel):
    intent_type: Literal["diem_query", "use_calculator", "chit_chat", "out_of_domain", "inappropriate", "needs_clarification"]
    tasks: List[SearchTask] = Field(default_factory=list)

class CriticEvaluation(BaseModel):
    is_satisfactory: bool = Field(description="True se la risposta fornisce le informazioni richieste, False altrimenti.")
    feedback: str = Field(description="Motivazione della bocciatura e suggerimento operativo.")

class ExtractedFilter(BaseModel):
    key: str = Field(description="Nome esatto del metadato presente nel manifest.")
    value: Union[str, List[str]] = Field(description="Uno o più valori esatti del metadato presente nel manifest, scritti in minuscolo.")

class Filter_Extraction(BaseModel):
    filters: List[ExtractedFilter] = Field(default_factory=list)


# Definizione AgentState, preso in input da tutti i nodi del grafo
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    plan: ExecutionPlan
    tool_results: str
    retry_count: int
    is_satisfactory: bool
    critic_feedback: Optional[str]

# Inizializzazione modelli

# 1. Componenti del Planner (Splittati in 3 LLM strutturati)
classifier_llm = ChatOllama(model="qwen3:14b", temperature=0).with_structured_output(QueryClassification)
reformulator_llm = ChatOllama(model="qwen3:14b", temperature=0).with_structured_output(QueryReformulation)
tool_selector_llm = ChatOllama(model="qwen3:14b", temperature=0).with_structured_output(ToolSelection)

# 2. Altri nodi del grafo
synthesizer_llm = ChatOllama(model="qwen3:14b", temperature=0)
critic_llm = ChatOllama(model="qwen3:14b", temperature=0).with_structured_output(CriticEvaluation)
filter_enricher_llm = ChatOllama(model="qwen3:14b", temperature=0).with_structured_output(Filter_Extraction)

# Funzioni 

STATIC_TOOL_OVERRIDES = {
    # aule che appartengono al dipartimento
    "aula infografica": "dipartimento",
    "aula didattica di base": "dipartimento",
}

def ultima_domanda_utente_e_storia(messages: list[BaseMessage], max_tokens: int = 4):
    indice_ultima_domanda = None
    for i in range(len(messages) - 1, -1, -1):
        if isinstance(messages[i], HumanMessage):
            indice_ultima_domanda = i
            break

    if indice_ultima_domanda is None:
        return "", []

    ultima_domanda = messages[indice_ultima_domanda].content
    storia_grezza = messages[:indice_ultima_domanda]
    storia = trim_messages(
        storia_grezza,
        max_tokens=max_tokens,
        token_counter=len,
        strategy="last",
        allow_partial=False,
    ) if storia_grezza else []
    return ultima_domanda, storia


def _build_metadata_condition(chiave_metadato: str, valore_metadato: Any):
    
    # should
    METADATI_TEXT = {
        "membri", "commission", "tipologia", "dipartimento",
        "nome_progetto", "responsabile", "titolo", "ufficio",
        "nominativo", "universita_partner", "destinatario",
        "project_period", "programma", "struttura", "questions"
    }


    path_metadato = f"metadata.{chiave_metadato}"
    valore_normalizzato = str(valore_metadato).lower()

    if chiave_metadato in METADATI_TEXT:
        return models.FieldCondition(
            key=path_metadato,
            match=models.MatchTextAny(text_any=valore_normalizzato)
        )

    return models.FieldCondition(
        key=path_metadato,
        match=models.MatchValue(value=valore_normalizzato)
    )


def _build_qdrant_filter(active_conditions: List[Any]):
    if not active_conditions:
        return None

    return models.Filter(must=active_conditions)


def _qdrant_search_named_vector(
    collection_name: str,
    vector_name: str,
    query_vector: List[float],
    qdrant_filter: Optional[Any],
    limit: int = QDRANT_LIMIT_PER_VECTOR
):
    response = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        using=vector_name,
        query_filter=qdrant_filter,
        limit=limit,
        with_payload=True,
        with_vectors=False
    )

    return response.points


def _point_to_document(point, matched_vector: str) -> Document:
    payload = point.payload or {}

    page_content = (
        payload.get("page_content")
        or payload.get("content")
        or payload.get("text")
        or ""
    )

    metadata = dict(payload.get("metadata", {}) or {})
    metadata["_qdrant_score"] = point.score
    metadata["_matched_vector"] = matched_vector

    return Document(page_content=page_content, metadata=metadata)


def _invoke_question_vectors_with_conditions(
    collection_name: str,
    query_testuale: str,
    active_conditions: List[Any],
    label: str,
    limit_per_vector: int = QDRANT_LIMIT_PER_VECTOR,
    top_k: int = TOP_K_FINAL
):
    qdrant_filter = _build_qdrant_filter(active_conditions)
    query_vector = embeddings.embed_query(query_testuale)

    if qdrant_filter:
        print(f"🔍 {label} sui vettori questions con filtro per: {query_testuale}")
        try:
            print(json.dumps(qdrant_filter.model_dump(), indent=2, ensure_ascii=False))
        except AttributeError:
            print(json.dumps(qdrant_filter.dict(), indent=2, ensure_ascii=False))
    else:
        print(f"🔍 {label} sui vettori questions senza filtro per: {query_testuale}")

    best_by_point_id = {}

    for vector_name in SEARCH_VECTOR_NAMES:
        points = _qdrant_search_named_vector(
            collection_name=collection_name,
            vector_name=vector_name,
            query_vector=query_vector,
            qdrant_filter=qdrant_filter,
            limit=limit_per_vector
        )

        for point in points:
            current_best = best_by_point_id.get(point.id)

            if current_best is None or point.score > current_best["score"]:
                best_by_point_id[point.id] = {
                    "point": point,
                    "score": point.score,
                    "matched_vector": vector_name
                }

    best_results = sorted(
        best_by_point_id.values(),
        key=lambda item: item["score"],
        reverse=True
    )[:top_k]

    return [
        _point_to_document(item["point"], item["matched_vector"])
        for item in best_results
    ]

def etichetta_link(link: str, contesto: str = "") -> str:
    """Crea un testo cliccabile comprensibile per una fonte."""
    testo = f"{contesto} {link}".lower()
    if "piano-di-studi" in testo:
        return "Pagina piano di studi"
    if "regolamenti-cds" in testo or link.lower().endswith(".pdf"):
        return "Regolamento/manifesto PDF"
    if "corsi.unisa.it" in testo:
        return "Pagina del corso"
    if "diem.unisa.it" in testo:
        return "Pagina DIEM"
    return "Fonte ufficiale"


def estrai_link_utili(testo: str, max_links: int = 5) -> list[tuple[str, str]]:
    """Estrae link unici con etichette leggibili, preservando l'ordine."""
    if not testo:
        return []

    links = []
    visti = set()
    for riga in testo.splitlines():
        for raw in re.findall(r"https?://[^\s\]\)>,]+", riga):
            link = raw.rstrip(".,;:)")
            if not link or link in visti:
                continue
            visti.add(link)
            links.append((etichetta_link(link, riga), link))
            if len(links) >= max_links:
                return links
    return links


def ripulisci_risposta(testo: str) -> str:
    """Rimuove formule tecniche che rompono la quarta parete."""
    sostituzioni = {
        "dal contesto fornito": "dalle informazioni disponibili",
        "nel contesto fornito": "nelle informazioni disponibili",
        "secondo il contesto fornito": "dalle informazioni disponibili",
        "dai documenti forniti": "dalle informazioni disponibili",
        "nei documenti forniti": "nelle informazioni disponibili",
        "dai documenti recuperati": "dalle informazioni disponibili",
        "nei documenti recuperati": "nelle informazioni disponibili",
        "secondo i documenti recuperati": "dalle informazioni disponibili",
        "dal database": "dalle informazioni disponibili",
        "nel database": "nelle informazioni disponibili",
        "database": "fonti disponibili",
        "qdrant": "le fonti disponibili",
        "tramite il tool": "",
        "dal tool": "dalle informazioni disponibili",
        "nel tool": "nelle informazioni disponibili",
        "tool": "fonti disponibili",
        "chunk": "informazioni",
        "tag xml": "",
        "retrieval": "recupero delle informazioni",
    }
    risultato = testo
    for vecchio, nuovo in sostituzioni.items():
        risultato = re.sub(vecchio, nuovo, risultato, flags=re.IGNORECASE)
    return risultato.strip()


def aggiungi_link_utili(risposta: str, tool_results: str) -> str:
    """Garantisce una sezione Link utili quando le fonti contengono URL."""
    risposta = ripulisci_risposta(risposta)
    links = estrai_link_utili(tool_results)
    if not links or "link util" in risposta.lower():
        return risposta

    elenco = "\n".join(f"- [{label}]({link})" for label, link in links)
    return f"{risposta}\n\nLink utili:\n{elenco}"


## Planner Node

In [35]:
def planner_node(state: AgentState):

    ultima_domanda, history_precedente = ultima_domanda_utente_e_storia(state["messages"], max_tokens=4)

    # COMPONENTE 1: CLASSIFICATORE DELLA QUERY
  
    prompt_classifier = SystemMessage(content=(
        "Sei il Classificatore di Intenti del dipartimento DIEM (Dipartimento Ingegneria dell'informazione ed elettrica e Matematica Applicata). Il tuo scopo è analizzare l'ultima domanda e classificarla.\n\n"
        "REGOLE DI CLASSIFICAZIONE (intent_type):\n"
        "- 'diem_query': Domande su iscrizioni, docenti, corsi, bandi, erasmus, iniziative o strutture.\n"
        "- 'use_calculator': Usalo SOLO se l'utente chiede il VOTO di LAUREA.\n"
        "- 'needs_clarification': La domanda è troppo generica o mancano i soggetti (es. 'orari?', 'chi è?') e la cronologia non aiuta.\n"
        "- 'chit_chat': Saluti o ringraziamenti.\n"
        "- 'out_of_domain': Sport, meteo, ricette o argomenti non universitari.\n"
        "- 'inappropriate': Offese o volgarità."
    ))

    messages_classifier = [prompt_classifier] + history_precedente + [HumanMessage(content=f"Richiesta utente: '{ultima_domanda}'")]
    classification = classifier_llm.invoke(messages_classifier)
    intent = classification.intent_type

    # Se l'intento non richiede una ricerca documentale, restituiamo subito il piano vuoto
    if intent != "diem_query":
        return {"plan": ExecutionPlan(intent_type=intent, tasks=[]), "critic_feedback": None}
    
    # Intercettiamo avviso feedback, potremmo essere in un retry
    avviso_feedback = ""
    if state.get("critic_feedback") and state.get("retry_count", 0) > 0:
        avviso_feedback = (
            f"\n\nREVISIONE DEL TENTATIVO PRECEDENTE (Il critico ha bocciato il piano):\n"
            f"Problema rilevato: '{state['critic_feedback']}'.\n"
            "Genera query di ricerca migliori ampliandole con sinonimi, nomi ufficiali, sigle o parole chiave aggiuntive."
        )

    # COMPONENTE 2: RIFORMULATORE DELLA QUERY

    prompt_reformulator = SystemMessage(content=(
        "Sei il Riformulatore di Query del dipartimento DIEM. Il tuo compito è trasformare la richiesta dell'utente in sotto-query ottimizzate per la ricerca semantica.\n\n"
        "REGOLE RIGIDE:\n"
        "1. FOCUS SULL'ATTUALITÀ: Il tuo obiettivo è rispondere SOLO all'ultimo messaggio dell'utente. "
        "   La cronologia ti serve ESCLUSIVAMENTE per capire il contesto, risolvere pronomi (es. 'esso', 'questo corso') o ellissi (es. 'e per la magistrale?').\n"
        "2. SCOMPOSIZIONE: Se l'utente ha fatto domande multiple o complesse, scomponile in query atomiche separate.\n"
        "3. NORMALIZZAZIONE TERMINOLOGICA:\n"
        "   - Ogni volta che compare 'digital medicine', riscrivilo come 'medicina digitale'.\n"
        "   - Ogni volta che compare 'digital energy', riscrivilo come 'energia digitale'.\n"
        "   - Se nella query c'è un acronimo lascialo inalterato, non lo estendere. Ad esempio DIEM=DIEM\n"
        "4. QUERY SEMANTICHE: Se la query dell'utente è già completa di significato e non ha usato pronomi per sostituire soggetti allora non modificarla ulteriormente.\n\n"
        "   Se decidi di modificarla, fai attenzione: \n"
        "5. PRONOMI: Risolvi e sostituisci sempre i pronomi (es. 'quali corsi insegna' -> 'corsi insegnati da [nome docente]').\n"
        "6. La query non deve essere un singolo termine o solo un nome proprio. Deve condensare l'intero intento della richiesta.\n"
        "7. INTEGRITÀ: NON inventare, NON aggiungere e NON inferire dettagli, corsi o nomi se non esplicitamente presenti nella richiesta o nella cronologia.\n"
        "8. IMPORTANTE: In caso di una singola query che contiene domande multiple fai una riformulazione per ogni singola query scomposta. Ma se la query chiede una cosa soltanto, fai al massimo 3 riformulazioni.\n"

    ))

    prompt_operativo_reformulator = HumanMessage(content=(
        f"Richiesta attuale dell'utente da ottimizzare: '{ultima_domanda}'{avviso_feedback}"
    ))

    messages_reformulator = [prompt_reformulator] + history_precedente + [prompt_operativo_reformulator]
    reformulation = reformulator_llm.invoke(messages_reformulator)

  
    # COMPONENTE 3: SELETTORE DEI TOOL 

    mappa_collezioni_stringa = "\n".join([
        f"- Nome Collezione: '{col['collection_name']}'\n  Descrizione: {col['description']}"
        for col in collections_config
    ])

    prompt_tool_selector = SystemMessage(content=(
        "Sei il Selettore dei Tool del dipartimento DIEM. Il tuo unico scopo è associare a ciascuna query ottimizzata la collezione corretta in cui effettuare la ricerca.\n\n"
        "=== MAPPA DELLE COLLEZIONI AMMESSE E DELLE LORO DESCRIZIONI ===\n"
        f"{mappa_collezioni_stringa}\n\n"
        "REGOLE RIGIDE:\n"
        "1. Scegli il 'tool_name' ESCLUSIVAMENTE tra i nomi delle collezioni sopra elencate.\n"
        "2. NON inventare mai nomi di collezioni (es. non usare 'personale', usa 'docenti').\n"
        "3. Analizza la descrizione di ogni collezione per capire qual è la più pertinente per la query."
    ))

    corpo_richiesta_tools = "Associa la collezione corretta a ciascuna di queste query ottimizzate:\n" + "\n".join([f"- {q}" for q in reformulation.queries])
    messages_tool_selector = [prompt_tool_selector, HumanMessage(content=corpo_richiesta_tools)]

    # Gestione collezioni da forzare in caso di parole chiave predefinite nella query

    assignments = []

    for query in reformulation.queries:
        query_lower = query.lower()

        forced_tool = None

        for keyword, tool_name in STATIC_TOOL_OVERRIDES.items():
            if keyword in query_lower:
                forced_tool = tool_name
                break

        if forced_tool:
            assignments.append(
                ToolAssignment(
                    query=query,
                    tool_name=forced_tool
                )
            )

    remaining_queries = [
        q for q in reformulation.queries
        if q not in [a.query for a in assignments]
    ]

    if remaining_queries:
        corpo_richiesta_tools = (
            "Associa la collezione corretta a ciascuna di queste query ottimizzate:\n"
            + "\n".join([f"- {q}" for q in remaining_queries])
        )

        messages_tool_selector = [
            prompt_tool_selector,
            HumanMessage(content=corpo_richiesta_tools)
        ]

        tool_selection = tool_selector_llm.invoke(messages_tool_selector)

        assignments.extend(tool_selection.assignments)

    tool_selection = ToolSelection(assignments=assignments)

    # Traduciamo l'output dei 3 componenti nel formato SearchTask atteso dal resto del grafo

    tasks_finali = []

    for assignment in tool_selection.assignments:
        tasks_finali.append(
            SearchTask(
            tool_name=assignment.tool_name,
            query=assignment.query,
            filters=[]
        ))

    return {"plan": ExecutionPlan(intent_type=intent, tasks=tasks_finali), "critic_feedback": None}

## Filter_Enricher Node

In [36]:
def filter_enricher_node(state: AgentState):

    plan = state.get("plan")

    if not plan or plan.intent_type != "diem_query" or not plan.tasks:
        return {"plan": plan}

    updated_tasks = []

    for task in plan.tasks:
        nome_collezione = task.tool_name
        testo_questions = task.query
        manifest_tool = manifests_map.get(nome_collezione, [])

        prompt_filtri = (
            "Sei un modulo specializzato nell'estrazione di metadati da una query.\n"
            "COMPITI:\n"
            "1. Estrai le parole chiave più importanti per popolare il metadato keywords.\n"
            "2. Estrai coppie chiave-valore usando SOLO i metadati e i corrsipondenti valori ammessi nel manifest.\n"
            "3. NON compilare mai metadati se NON sono esplicitamente espresse nella query. \n"
            "4. I valori stringa devono essere in minuscolo.\n"
            "5. Se nella query non è presente un valore esatto ammesso dal manifest, NON creare quel filtro."
                "Non usare mai \"null\", null, \"none\", \"non specificato\" o valori placeholder."
            "6. NON creare mai più di un valore per metadato\n"
            f"Query da analizzare: \"{testo_questions}\"\n\n"
            f"=== MANIFEST METADATI AMMESSI PER IL TOOL '{nome_collezione.upper()}' ===\n"
            f"{json.dumps(manifest_tool, indent=2, ensure_ascii=False)}\n"
        )

        risultato_llm = filter_enricher_llm.invoke(prompt_filtri)

        filtri_processati = []
        for f in risultato_llm.filters:
            operatore = METADATA_CLAUSE_MAPPING.get(f.key, "should")

            filtri_processati.append({
                "key": f.key,
                "value": f.value,
                "operator": operatore
            })

        task.query = testo_questions
        task.filters = filtri_processati

        updated_tasks.append(task)

    plan.tasks = updated_tasks
    
    return {"plan": plan}

## Executor Node

In [37]:
def executor_node(state: AgentState):
    """Esegue le ricerche su Qdrant usando filtri MUST rigidi e SHOULD con fallback."""
    plan = state.get("plan")

    if not plan or plan.intent_type != "diem_query" or not plan.tasks:
        return {"tool_results": "Nessuna ricerca necessaria."}

    risultati_aggregati = []

    with qdrant_lock:
        for task in plan.tasks:
            nome_collezione = task.tool_name
            query_testuale = task.query 

            query_testuale=query_testuale.lower()

            if nome_collezione not in retrievers_map:
                risultati_aggregati.append(f"Errore: Il tool {nome_collezione} non esiste.")
                continue

            must_conditions = []
            optional_conditions = []

            for f in task.filters:
                condition = _build_metadata_condition(f["key"], f["value"])
                if condition is None:
                    continue

                if f.get("operator") == "must":
                    must_conditions.append(condition)
                else:
                    optional_conditions.append({
                        "key": f["key"],
                        "condition": condition
                    })

            docs = _invoke_question_vectors_with_conditions(
                nome_collezione,
                query_testuale,
                must_conditions,
                f"Ricerca base su '{nome_collezione}'"
            )

            active_optional_conditions = []

            if docs:
                for optional_filter in optional_conditions:
                    trial_conditions = must_conditions + active_optional_conditions + [optional_filter["condition"]]

                    trial_docs = _invoke_question_vectors_with_conditions(
                        nome_collezione,
                        query_testuale,
                        trial_conditions,
                        f"Tentativo SHOULD '{optional_filter['key']}' su '{nome_collezione}'"
                    )

                    if trial_docs:
                        active_optional_conditions.append(optional_filter["condition"])
                        docs = trial_docs
                    else:
                        print(
                            f"ℹ️ Filtro SHOULD '{optional_filter['key']}' ignorato: "
                            "avrebbe eliminato tutti i risultati."
                        )

            if docs:
                estratti = []
                for i, d in enumerate(docs, start=1):
                    metadata = d.metadata or {}

                    # Estrazione sicura del solo link in "source"
                    fonte = str(metadata.get("source") or "").strip()
                    sezione = str(metadata.get("section" or "")).strip()
                    sottosezione = str(metadata.get("subsection" or "")).strip()

                    # Recupero i dati di debug di Qdrant
                    score = metadata.get("_qdrant_score", 0.0)
                    matched_vector = metadata.get("_matched_vector", "sconosciuto")

                    # Costruzione del blocco di testo formattato
                    righe = [f"Informazione {i} [score={score:.4f}, vettore={matched_vector}]:"]

                    if fonte and fonte.lower() not in {"none", "null"}:
                        righe.append(f"Fonte/Link: {fonte}")

                    if sezione and sezione.lower() not in {"none", "null"}:
                        righe.append(f"Sezione: {sezione}")

                    if sottosezione and sottosezione.lower() not in {"none", "null"}:
                        righe.append(f"sottosezione: {sottosezione}")

                    righe.append(f"Testo utile: {d.page_content.strip()}")
                    estratti.append("\n".join(righe))

                testo_estratti = "\n\n".join(estratti)
            else:
                testo_estratti = "Nessun risultato trovato."

            risultati_aggregati.append(
                f"--- RISULTATI DA '{nome_collezione}' PER LA QUERY '{query_testuale}' ---\n{testo_estratti}"
            )

    return {"tool_results": "\n\n".join(risultati_aggregati)}

## Synthetizer Node

In [38]:
def synthesizer_node(state: AgentState):
    plan = state.get("plan")
    tool_results = state.get("tool_results", "")

    # GESTIONE SICUREZZA E CHIT-CHAT (Bypassiamo l'LLM o gli diamo un prompt specifico)
    if plan.intent_type == "inappropriate":
        return {"messages": [AIMessage(content="Il tuo linguaggio non è appropriato. Sono qui per aiutarti con le informazioni sul dipartimento DIEM. Come posso esserti utile in merito?")], "retry_count": state.get("retry_count", 0)}

    elif plan.intent_type == "out_of_domain":
        return {"messages": [AIMessage(content="Mi dispiace, ma come assistente del DIEM posso rispondere solo a domande relative al nostro dipartimento, ai corsi di laurea, ai docenti e alle procedure amministrative dell'Università di Salerno.")], "retry_count": state.get("retry_count", 0)}

    elif plan.intent_type == "chit_chat":
        prompt_synth = SystemMessage(content="Sei l'assistente del DIEM dell'Università di Salerno. Rispondi al saluto in modo cortese, breve e professionale.")
        response = synthesizer_llm.invoke([prompt_synth, state["messages"][-1]])
        return {"messages": [response], "retry_count": state.get("retry_count", 0)}

    elif plan.intent_type == "needs_clarification":
        # Passiamo la palla all'LLM affinché formuli una richiesta di chiarimento sensata
        prompt_synth = SystemMessage(content=(
            "Sei l'assistente del DIEM. L'ultima richiesta dell'utente è troppo vaga o incompleta per poter fare una ricerca nei database istituzionali. "
            "Chiedigli gentilmente e brevemente di specificare meglio cosa sta cercando (es. il nome del docente, a quale corso di laurea si riferisce, ecc.)."
        ))
        response = synthesizer_llm.invoke([prompt_synth, state["messages"][-1]])
        return {"messages": [response], "retry_count": state.get("retry_count", 0)}

    elif plan.intent_type == "use_calculator":
        risposta = "⚙️ [TRIGGER_CALCOLATORE]\nHo appena sbloccato il calcolatore per te! Compila i tuoi dati nel modulo che è appena comparso qui a destra 👉"
        return {"messages": [AIMessage(content=risposta, id="calc_msg_123")], "retry_count": state.get("retry_count", 0)}


    else:

        ultima_domanda, history_precedente = ultima_domanda_utente_e_storia(state["messages"], max_tokens=4)

        prompt_synth = SystemMessage(content=(
            "Sei l'assistente virtuale ufficiale del DIEM (Dipartimento Ingegneria dell'informazione ed elettrica e Matematica Applicata) dell'Università di Salerno. "
            "Rispondi in italiano con tono professionale, naturale e utile.\n"
            "Usa solo le informazioni ufficiali disponibili nel messaggio dell'utente: non inventare nomi, date, email, link, scadenze o procedure.\n"
            "Non copiare lunghi blocchi di testo: rielabora le informazioni in forma discorsiva, chiara e orientata alla domanda.\n"
            "Preserva sempre le informazioni strutturate quando l'utente le richiede o quando sono centrali per rispondere: elenchi, liste, tabelle, insegnamenti, piani di studio, bandi, requisiti, scadenze, contatti, membri di organi/commissioni, ruoli, email, CFU, anni di corso, curricula e link.\n"
            "In questi casi non sostituire l'elenco con una descrizione generale: riporta gli elementi disponibili in elenco o tabella markdown compatta, raggruppandoli per categoria/anno/curriculum/ruolo quando l'informazione e presente.\n"
            "Se le informazioni recuperate sono parziali, dichiaralo in modo naturale e riporta comunque tutti gli elementi utili trovati.\n"
            "Se la risposta contiene molti dettagli, organizza in brevi paragrafi, punti essenziali o tabelle markdown leggibili.\n"
            "Se una parte della domanda non trova risposta, dillo con una frase naturale e poi fornisci comunque le informazioni utili trovate.\n"
            "Non nominare mai tool, database, Qdrant, chunk, tag XML, retrieval, contesto fornito o meccanismi interni.\n"
            "Quando nelle informazioni disponibili sono presenti URL o fonti, includi sempre una sezione finale 'Link utili:' con i link pertinenti (non metterne più di due).\n"
            "Evita formule come 'dal contesto fornito', 'nei documenti recuperati' o simili."
        ))

        prompt_utente_con_contesto = HumanMessage(content=(
            f"INFORMAZIONI UFFICIALI DISPONIBILI:\n"
            f"{tool_results}\n\n"
            f"DOMANDA DELL'UTENTE:\n{ultima_domanda}\n\n"
            "Rispondi alla domanda in modo diretto e discorsivo, usando solo le informazioni ufficiali disponibili sopra."
        ))


        # La nuova array di messaggi: Istruzioni -> Vecchia chat -> Nuova Domanda + Dati Qdrant
        messages_to_invoke = [prompt_synth] + history_precedente + [prompt_utente_con_contesto]

        response = synthesizer_llm.invoke(messages_to_invoke)
        response = AIMessage(
            content=aggiungi_link_utili(response.content, tool_results),
            id=getattr(response, "id", None)
        )
        return {"messages": [response], "retry_count": state.get("retry_count", 0)}

## Critic Node

In [39]:
MAX_CRITIC_RETRIES = 2

def critic_node(state: AgentState):
    """Valuta la qualità e se serve forza un re-try."""

    plan = state.get("plan")
    if plan and plan.intent_type != "diem_query":
        return {"is_satisfactory": True, "critic_feedback": None}

    messages = state["messages"]
    last_msg = messages[-1] # Questa è l'AIMessage generata dal synthesizer
    last_response = last_msg.content
    response_original = last_msg.content or ""
    current_retries = state.get("retry_count", 0)

    # Trova l'ultima vera domanda dell'utente
    user_question = "Sconosciuta"
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            user_question = msg.content
            break

    response_to_evaluate = ripulisci_risposta(response_original)
    response_was_cleaned = response_to_evaluate != response_original.strip()

    critic_prompt = f"""
    Sei un revisore di qualità per un assistente universitario. Il tuo unico scopo è verificare la congruenza logica tra domanda e risposta.


    ULTIMA DOMANDA UTENTE: {user_question}
    RISPOSTA GENERATA DA VALUTARE: {last_response}

    REGOLE:
    1. Approva se la risposta copre il nucleo della domanda, anche se potrebbe essere scritta meglio.
    2. Approva se la risposta ammette con chiarezza che una parte dell'informazione non e disponibile, ma fornisce cio che ha trovato.
    3. Boccia solo se la risposta e fuori tema, contraddice la domanda, inventa dati evidenti o ignora completamente una richiesta principale.
    4. Non bocciare per assenza di link, stile non perfetto, sintesi breve o formulazione migliorabile.
    5. Se bocci, dai un feedback operativo breve: quale entita o aspetto cercare meglio.
    """


    evaluation = critic_llm.invoke(critic_prompt)

    if evaluation.is_satisfactory:
        if response_was_cleaned:
            return {
                "is_satisfactory": True,
                "messages": [AIMessage(content=response_to_evaluate, id=last_msg.id)],
                "critic_feedback": None,
            }
        return {"is_satisfactory": True, "critic_feedback": None}

    if current_retries < MAX_CRITIC_RETRIES:
        return {
            "is_satisfactory": False,
            "critic_feedback": evaluation.feedback,
            "retry_count": current_retries + 1,
        }

    # Uscita di sicurezza: dopo il retry non trasformiamo la risposta in un fallimento generico.
    # Manteniamo l'ultima risposta disponibile, ripulita da eventuali formule tecniche.
    final_response = response_to_evaluate.strip() or "Mi dispiace, non ho trovato informazioni sufficienti per rispondere con precisione a questa domanda."
    return {
        "is_satisfactory": True,
        "messages": [AIMessage(content=final_response, id=last_msg.id)],
        "retry_count": current_retries,
        "critic_feedback": None,
    }

## Workflow

In [40]:
# --- D. ROUTING E COMPILAZIONE DEL GRAFO ---

def route_after_critic(state: AgentState):
    if state.get("is_satisfactory", False):
        return END
    return "planner"

workflow = StateGraph(AgentState)

workflow.add_node("planner", planner_node)
workflow.add_node("filter_enricher", filter_enricher_node)
workflow.add_node("executor", executor_node)
workflow.add_node("synthesizer", synthesizer_node)
workflow.add_node("critic", critic_node)

workflow.set_entry_point("planner")

# Sequenza logica e lineare
workflow.add_edge("planner", "filter_enricher")
workflow.add_edge("filter_enricher", "executor")
workflow.add_edge("executor", "synthesizer")
workflow.add_edge("synthesizer", "critic")
workflow.add_conditional_edges("critic", route_after_critic)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)
print("Grafo Plan-and-Execute compilato con successo!")

Grafo Plan-and-Execute compilato con successo!


# Interfaccia GRADIO

## Tool Calcolatrice

In [41]:
def calcola_voto_laurea(dati) -> str:
    media = dati.media
    ordinamento = dati.ordinamento
    ciclo = dati.ciclo
    PT = dati.punti_tesi
    PC = dati.punti_carriera or 0.0

    if PT is None:
        return "⚠️ **Errore:** Il punteggio della tesi è obbligatorio. Inserisci un valore nel modulo a destra per effettuare il calcolo."

    risultato = f"📊 **Simulazione Voto di Laurea ({ciclo.capitalize()}, Ordinamento {ordinamento.capitalize()})**\n"
    risultato += f"📈 **Media Ponderata di partenza:** {media}\n\n"

    if ordinamento == "vecchio":
        V_base_raw = (4.1 * media) - (7.8 if ciclo == "triennale" else 8.8)
        V_base = math.floor(V_base_raw + 0.5)
        max_punti = 5 if ciclo == "triennale" else 6

        risultato += f"Voto Base (arrotondato): **{V_base}**\n"
        punti_tot = min(PT + PC, max_punti)
        voto_fin = V_base + punti_tot
        lode_text = " (🎉 **Lode possibile!**)" if voto_fin >= 112 else ""
        risultato += f"Punti Tesi + Carriera: {PT + PC} (Max applicabile: {max_punti})\n"
        risultato += f"🎓 **Voto Finale Stimato:** {min(voto_fin, 110)}{lode_text}"

    else:
        V_MIN = (media * 110) / 30
        limite = 5 if ciclo == "triennale" else 6
        divisore_pt = 2 if ciclo == "triennale" else 4
        sottraendo_fc = 7.8 if ciclo == "triennale" else 8.8

        risultato += f"Voto Minimo Base (V_MIN): **{round(V_MIN, 2)}**\n"

        def calcola_formula(pt):
            pcp = PC * (pt / divisore_pt)
            fcp = ((4.1 * media - sottraendo_fc) - V_MIN) * (pt / divisore_pt)
            voto_finale = V_MIN + min(pt + pcp, limite) + fcp
            return math.floor(voto_finale + 0.5)

        v_fin = calcola_formula(PT)
        lode_text = " (🎉 **Lode possibile!**)" if v_fin >= 112 else ""
        risultato += f"🎓 **Voto Finale Stimato:** {min(v_fin, 110)}{lode_text}"

    return risultato + "\n\n*(Nota: la lode è attribuibile solo se il punteggio calcolato è maggiore o uguale a 112 ed è presente l'unanimità della commissione.)*"


def calcola_da_form(ciclo, ordinamento, media, carriera, tesi, history):
    class DatiForm:
        def __init__(self, c, o, m, pc, pt):
            self.ciclo = c
            self.ordinamento = o
            self.media = m
            self.punti_carriera = pc
            self.punti_tesi = pt

    voto_tesi = float(tesi) if (tesi is not None and tesi != "") else None
    dati = DatiForm(ciclo, ordinamento, float(media), float(carriera), voto_tesi)

    risultato_calcolo = calcola_voto_laurea(dati)
    info_tesi = f"{tesi} punti" if voto_tesi is not None else "Non inserito"
    messaggio_utente = (
        f"🧮 **Richiesta Calcolo tramite Form:**\n"
        f"- Percorso: {ciclo.upper()} ({ordinamento.upper()})\n"
        f"- Media: {media} | Bonus: {carriera} | Tesi: {info_tesi}"
    )

    if history is None:
        history = []

    history.append({"role": "user", "content": messaggio_utente})
    history.append({"role": "assistant", "content": risultato_calcolo})
    return history


## CSS e Avvio GRADIO

In [42]:
config_memoria = {"configurable": {"thread_id": "sessione_diem_plan_1"}}

def rispondi(messaggio_utente, history):
    inizio_timer = time.time()

    if history is None:
        history = []

    history.append({"role": "user", "content": messaggio_utente})
    history.append({"role": "assistant", "content": "*Elaborazione in corso...*"})

    yield "", history, gr.update(), "<small style='color: #0284c7; font-weight: 600;'>⏳ *Elaborazione richiesta in corso...*</small>"

    log_di_sistema = "🧠 **Processo Logico dell'Agente:**\n"
    testo_risposta = ""
    mostra_calcolatore = False

    def aggiorna_ultimo(testo_nuovo):
        nonlocal mostra_calcolatore
        if "[TRIGGER_CALCOLATORE]" in testo_nuovo:
            mostra_calcolatore = True
            testo_nuovo = testo_nuovo.replace("⚙️ [TRIGGER_CALCOLATORE]\n", "")

        if isinstance(history[-1], dict):
            history[-1]["content"] = testo_nuovo
        else:
            history[-1].content = testo_nuovo

    for update in app.stream(
        {
            "messages": [HumanMessage(content=messaggio_utente)],
            "retry_count": 0,
            "critic_feedback": None
        },
        config=config_memoria,
        stream_mode="updates"
    ):
        tempo_parziale = round(time.time() - inizio_timer, 1)
        testo_timer = f"<small style='color: #0284c7; font-weight: 600;'>⏳ *Elaborazione in corso... ({tempo_parziale}s)*</small>"

        for nodo_corrente, stato_attuale in update.items():

            if nodo_corrente == "planner":
                piano = stato_attuale.get("plan")
                if piano and piano.intent_type != "diem_query":
                    log_di_sistema += f"- 💬 *Intento rilevato:* `{piano.intent_type}`. Nessun tool di ricerca richiesto.\n"
                elif piano and piano.tasks:
                    log_di_sistema += f"- 📋 *Piano generato:* Scomposta la domanda in **{len(piano.tasks)}** task di ricerca.\n"
                    for i, task in enumerate(piano.tasks):
                        log_di_sistema += f"   {i+1}. Cerco in `{task.tool_name}`: *'{task.query}'*\n"

                aggiorna_ultimo(log_di_sistema + "\n---\n*Elaborazione in corso...*")
                yield "",list(history), gr.update(visible=True) if mostra_calcolatore else gr.update(), testo_timer

            elif nodo_corrente == "executor":
                log_di_sistema += "- ⚙️ *Dati estratti dai documenti.*\n"
                aggiorna_ultimo(log_di_sistema + "\n---\n*Elaborazione in corso...*")
                yield "",list(history), gr.update(visible=True) if mostra_calcolatore else gr.update(), testo_timer

            elif nodo_corrente == "synthesizer":
                messaggi = stato_attuale.get("messages", [])
                if messaggi and isinstance(messaggi[-1], AIMessage):
                    testo_risposta = messaggi[-1].content
                    log_di_sistema += "- ✍️ *Risposta sintetizzata.*\n"
                    aggiorna_ultimo(log_di_sistema + "\n---\n" + testo_risposta)

                    tempo_finale = round(time.time() - inizio_timer, 2)
                    testo_timer_finale = f"<small style='color: #10b981; font-weight: 600;'>✅ Risposta generata in **{tempo_finale}** secondi.</small>"
                    yield "", list(history), gr.update(visible=True) if mostra_calcolatore else gr.update(), testo_timer_finale

            elif nodo_corrente == "critic":
                messaggi = stato_attuale.get("messages", [])
                if messaggi and isinstance(messaggi[-1], AIMessage):
                    testo_risposta = messaggi[-1].content

                if not stato_attuale.get("is_satisfactory", True):
                    log_di_sistema += "- 🧐 *Il Critico ha bocciato la risposta. Correggo...*\n"
                    aggiorna_ultimo(log_di_sistema + "\n---\n*[Cerco info più precise...]*")
                    yield "", list(history), gr.update(visible=True) if mostra_calcolatore else gr.update(), testo_timer
                else:
                    log_di_sistema += "- ✅ *Verifica superata.*\n"
                    aggiorna_ultimo(log_di_sistema + "\n---\n" + testo_risposta)

                    tempo_finale = round(time.time() - inizio_timer, 2)
                    testo_timer_finale = f"<small style='color: #10b981; font-weight: 600;'>✅ Risposta verificata e generata in **{tempo_finale}** secondi.</small>"
                    yield "", list(history), gr.update(visible=True) if mostra_calcolatore else gr.update(), testo_timer_finale


def pulisci_tutto():
    global config_memoria
    nuovo_thread_id = f"sessione_diem_{uuid.uuid4().hex[:8]}"
    config_memoria = {"configurable": {"thread_id": nuovo_thread_id}}
    return [], "", "<small>✨ Chat svuotata con successo. Pronto per nuove domande!</small>"

MESSAGGIO_BENVENUTO = [
    {
        "role": "assistant",
        "content": "👋 **Ciao! Sono l'Assistente Virtuale del DIEM.**\n\nPuoi chiedermi informazioni sui professori, i requisiti di laurea, i corsi, oppure usare gli strumenti integrati per simulare il tuo voto di laurea. Come posso aiutarti oggi?"
    }
]


tema_diem = gr.themes.Soft(
    primary_hue="sky",
    secondary_hue="blue",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Montserrat"), "ui-sans-serif", "system-ui", "sans-serif"]
)

css_custom = """
@import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@400;500;600;700&display=swap');

.gradio-container {
    font-family: 'Montserrat', sans-serif !important;
    background: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%);
}

.contain {
    max-width: 1100px !important;
    margin: 30px auto !important;
    box-shadow: 0 25px 50px -12px rgba(2, 132, 199, 0.15);
    border-radius: 24px;
    padding: 30px;
    background-color: #ffffff;
    border: 1px solid rgba(255, 255, 255, 0.6);
}

.header-container {
    display: flex;
    align-items: center;
    gap: 15px;
    margin-bottom: 10px;
    padding-bottom: 15px;
    border-bottom: 2px solid #f1f5f9;
}
.header-logo {
    height: 45px !important;
    border-radius: 6px;
}

.box-chat-personalizzato {
    background-color: #f8fafc !important;
    border-radius: 16px !important;
    border: 1px solid #e2e8f0 !important;
    box-shadow: inset 0 2px 4px rgba(0,0,0,0.02) !important;
    margin-top: 5px !important;
}

.pannello-calc-box {
    background-color: #f8fafc !important;
    border-radius: 16px !important;
    padding: 20px !important;
    border: 1px solid #e2e8f0 !important;
    margin-top: 5px !important;
}

/* Nasconde la barra nativa di Gradio (Copia, Condividi, Cestino) per non creare confusione */
.message-wrap > div > div > button { display: none !important; }
.share-button { display: none !important; }


/* === STILE BOLLE CHAT === */
.message-wrap .message {
    max-width: 80% !important;
    border-radius: 18px !important;
    line-height: 1.5 !important;
    padding: 12px 18px !important;
    font-size: 0.95em !important;
}
/* Messaggio dell'Utente (Destra - Colore pastello morbido) */
.message-wrap .user {
    background: linear-gradient(135deg, #bae6fd 0%, #e0f2fe 100%) !important; /* Azzurro chiarissimo */
    color: #0369a1 !important; /* Testo blu scuro per un contrasto elegante */
    border: 1px solid #7dd3fc !important; /* Bordino delicato azzurro */
    border-bottom-right-radius: 4px !important;
    box-shadow: 0 4px 8px rgba(2, 132, 199, 0.08) !important; /* Ombra leggerissima */
}
.message-wrap .bot {
    background-color: #ffffff !important;
    color: #334155 !important;
    border-bottom-left-radius: 4px !important;
    box-shadow: 0 2px 5px rgba(0,0,0,0.05) !important;
    border: 1px solid #e2e8f0 !important;
}

/* === STILE BOTTONI PROPORZIONATI E ALLINEAMENTO === */
button {
    border-radius: 10px !important;
    font-weight: 600 !important;
    transition: all 0.2s ease !important;
}
.tool-btn {
    background-color: #f8fafc !important;
    border: 1px solid #cbd5e1 !important;
    color: #334155 !important;
}
.tool-btn:hover { background-color: #f1f5f9 !important; border-color: #94a3b8 !important;}

.danger-btn {
    background-color: #fff1f2 !important;
    border: 1px solid #fecdd3 !important;
    color: #e11d48 !important;
}
.danger-btn:hover { background-color: #ffe4e6 !important; }

.invia-btn {
    background: linear-gradient(135deg, #0284c7 0%, #0ea5e9 100%) !important;
    border: none !important;
    color: white !important;
    box-shadow: 0 4px 10px rgba(2, 132, 199, 0.25) !important;
}
.invia-btn:hover { transform: translateY(-2px); filter: brightness(1.05); }

input.scroll-hide {
    border-radius: 12px !important;
}

/* NUOVE REGOLE DI IMPAGINAZIONE SENZA ERRORI DI SCALE */
.bottoni-destra { justify-content: flex-end !important; }
.bottoni-centro { justify-content: center !important; }
"""

esempi_domande = [
    "Chi è il professore Antonio Greco?",
    "Quali sono i requisiti di immatricolazione per la Magistrale in Ingegneria Informatica?",
    "Voglio calcolare il mio voto di laurea",
    "L'università ha collaborazioni Erasmus o di Doppio Titolo?"
]

html_header = """
<div class="header-container">
    <img src="https://www.diem.unisa.it/rescue/img/logo_standard.png" class="header-logo" alt="Logo">
    <h2 style="margin: 0; color: #0284c7; font-weight: 700; letter-spacing: -0.5px;">Assistente Virtuale DIEM</h2>
</div>
"""

with gr.Blocks(theme=tema_diem, css=css_custom) as interfaccia:

    gr.HTML(html_header)

    with gr.Row():

        # ==========================================
        # COLONNA 1: CHATBOT E COMANDI (Sinistra)
        # ==========================================
        with gr.Column(scale=2):

            # Applicata la classe bottoni-destra per spingerli automaticamente senza usare gr.Markdown
            with gr.Row(elem_classes=["bottoni-destra"]):
                btn_apri_calcolatore = gr.Button(" 🎓 Apri Calcolatore", elem_classes=["tool-btn"], scale=0, min_width=180)
                btn_pulisci = gr.Button(" 🗑️ Svuota Chat", elem_classes=["danger-btn"], scale=0, min_width=150)

            chatbot = gr.Chatbot(
                value=MESSAGGIO_BENVENUTO,
                height=480,
                show_label=False,
                avatar_images=(
                    "https://cdn-icons-png.flaticon.com/512/4140/4140048.png",
                    "https://cdn-icons-png.flaticon.com/512/8036/8036730.png"
                ),
                elem_classes=["box-chat-personalizzato"],
                layout="bubble",
 
            )
            indicatore_tempo = gr.Markdown("<small style='color: #64748b;'>In attesa di domande...</small>")

            with gr.Row():
                textbox_messaggio = gr.Textbox(
                    placeholder="Scrivi qui la tua domanda (es. 'Voglio calcolare il voto')...",
                    show_label=False,
                    scale=5
                )
                btn_invia = gr.Button("Invia ➔", variant="primary", scale=1, elem_classes=["invia-btn"])

            gr.Examples(examples=esempi_domande, inputs=textbox_messaggio, cache_examples=False)


        with gr.Column(scale=1, visible=False, elem_classes=["pannello-calc-box"]) as pannello_calcolatore:

            with gr.Row():
                gr.Markdown("Simulatore voto di Laurea")
                btn_chiudi_calcolatore = gr.Button("❌", size="sm", scale=0, min_width=100, elem_classes=["danger-btn"])

            gr.Markdown("<small style='color: #64748b;'>Compila i campi per una simulazione esatta basata sui regolamenti attuali.</small>")

            dropdown_ciclo = gr.Dropdown(choices=["triennale", "magistrale"], label="Ciclo di Studi", value="triennale", interactive=True)
            dropdown_ordinamento = gr.Dropdown(choices=["nuovo", "vecchio"], label="Ordinamento", info="Nuovo = dal 2025/26", value="nuovo", interactive=True)
            number_media = gr.Number(label="Media Ponderata", info="Usa il punto per i decimali (es. 25.45)", minimum=18.0, maximum=30.0, step=0.01, value=24.0, interactive=True)

            with gr.Row():
                number_carriera = gr.Number(label="Bonus Carriera", minimum=0.0, maximum=6.0, step=0.5, value=0.0, interactive=True, scale=1)
                number_tesi = gr.Number(label="Punti Tesi", minimum=0.0, maximum=10.0, step=0.5, value=1.0, interactive=True, scale=1)

            # Applicata la classe bottoni-centro per non stirare il pulsante
            with gr.Row(elem_classes=["bottoni-centro"]):
                btn_esegui_calcolo = gr.Button("Calcola Voto", elem_classes=["invia-btn"], scale=0, min_width=200)


    btn_apri_calcolatore.click(fn=lambda: gr.update(visible=True), inputs=None, outputs=[pannello_calcolatore])
    btn_chiudi_calcolatore.click(fn=lambda: gr.update(visible=False), inputs=None, outputs=[pannello_calcolatore])

    textbox_messaggio.submit(
        fn=rispondi,
        inputs=[textbox_messaggio, chatbot],
        outputs=[textbox_messaggio, chatbot, pannello_calcolatore, indicatore_tempo]
    )

    btn_invia.click(
        fn=rispondi,
        inputs=[textbox_messaggio, chatbot],
        outputs=[textbox_messaggio, chatbot, pannello_calcolatore, indicatore_tempo]
    )

    btn_esegui_calcolo.click(
        fn=calcola_da_form,
        inputs=[dropdown_ciclo, dropdown_ordinamento, number_media, number_carriera, number_tesi, chatbot],
        outputs=[chatbot]
    )

    btn_pulisci.click(
        fn=pulisci_tutto,
        inputs=None,
        outputs=[chatbot, textbox_messaggio, indicatore_tempo]
    )

print("Avvio interfaccia...")
interfaccia.launch(debug=True)

/var/folders/qq/s2n5tc_d7lbd3lfpr757lrxw0000gn/T/ipykernel_3105/2699045485.py:237: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=tema_diem, css=css_custom) as interfaccia:


Avvio interfaccia...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


KeyError: '_oh'